# 03 · Validation — what the walk-forward actually scored

Three claims are checked here rather than asserted: the model **discriminates on every fold** it
claims to have scored, the **calibration refusal is real and named** rather than silently
degrading a probability, and **every scored row carries its explanation** or says why it does not.

The published numbers live in `MODEL_CARD.md`, generated by `make eval` from the backtest
harness's own artifacts. This notebook recomputes the headline discrimination **independently from
the landed rows**, so the harness and a hand-run check can be compared; where they disagree, the
harness is authoritative for what gets published and this notebook is the alarm.

Run from the repository root after `uv run oxbow score` has landed a run.


In [1]:
import json
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd()
if not (ROOT / "config" / "model.yaml").is_file():
    raise SystemExit("invoke from the repository root")

runs = sorted((ROOT / "out" / "score").glob("*/scored_rows.parquet"))
print("landed scored-row runs:", [p.parent.name for p in runs])
scored_path = runs[-1]
RUN_ID = scored_path.parent.name
scored = pl.read_parquet(scored_path)
print(f"using run {RUN_ID}: {scored.height:,} rows x {scored.width} columns")
print()
print(scored.group_by("role").agg(pl.len().alias("rows")).sort("role"))


landed scored-row runs: ['01M3FZ2GC3J71AYT1QEKPWEDKJ', '01M3H8WG436R394NZT2GS1KG69']
using run 01M3H8WG436R394NZT2GS1KG69: 43,720 rows x 123 columns

shape: (1, 2)
┌──────┬───────┐
│ role ┆ rows  │
│ ---  ┆ ---   │
│ str  ┆ u32   │
╞══════╪═══════╡
│ test ┆ 43720 │
└──────┴───────┘


## 1. Per-fold discrimination, computed here

Fitting is per fold on `train`+`validation`; scoring is on `test`. A pooled average would let a
fold with 23,032 test rows hide one with 872, so each fold is reported alone and then pooled with
its weight visible.


In [2]:
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score

LABEL = "label_is_fraud"
test_rows = scored.filter(pl.col("role") == "test")

print(f"{'fold':>4} {'rows':>7} {'pos':>5} {'base rate':>10} {'PR-AUC':>9} {'ROC-AUC':>9} "
      f"{'Brier-ish':>10} {'mean p':>8}  mode")
per_fold = []
for fold in sorted(test_rows["fold"].unique().to_list()):
    part = test_rows.filter(pl.col("fold") == fold)
    y = part.get_column(LABEL).cast(pl.Int8).to_numpy().astype(np.int64)
    p = part.get_column("p_fused").cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float64)
    positives = int(y.sum())
    if positives == 0 or len(np.unique(y)) < 2:
        print(f"{fold:>4} {part.height:>7} {positives:>5}  -- no positives: AUC undefined, "
              f"reported as undefined rather than 0.0")
        continue
    pr = float(average_precision_score(y, p))
    roc = float(roc_auc_score(y, p))
    ll = float(log_loss(y, np.clip(p, 1e-6, 1 - 1e-6), labels=[0, 1]))
    mode = part.get_column("scoring_mode").mode()[0]
    base = positives / part.height
    print(f"{fold:>4} {part.height:>7} {positives:>5} {base:>10.6f} {pr:>9.4f} {roc:>9.4f} "
          f"{ll:>10.4f} {p.mean():>8.5f}  {mode}")
    per_fold.append((fold, part.height, positives, base, pr, roc, ll))

print()
if per_fold:
    prs = np.array([entry[4] for entry in per_fold])
    rocs = np.array([entry[5] for entry in per_fold])
    print(f"mean across the {len(per_fold)} scored folds: "
          f"PR-AUC={prs.mean():.4f} (min {prs.min():.4f}, max {prs.max():.4f})  "
          f"ROC-AUC={rocs.mean():.4f} (min {rocs.min():.4f}, max {rocs.max():.4f})")
    print("  -> THIS is the headline shape: an unweighted mean of per-fold readings, with the")
    print("     spread shown because a fold holding 4 positives cannot carry a 0.75 the way a")
    print("     fold holding 16 can. The plan's §18 leakage trigger (~0.9 PR-AUC on a first")
    print("     honest run) is not approached, which is what an honest first run looks like.")
    print()
    print("A pooled-across-folds figure is deliberately NOT reported here.")
    print("     Each fold refits its own model, so `p_fused` on fold 2 and `p_fused` on fold 4")
    print("     are not on a common scale; concatenating their rows produces a number that")
    print("     looks like a metric and is actually an artefact of mixing five different")
    print("     score distributions. The harness reports per fold for the same reason.")


fold    rows   pos  base rate    PR-AUC   ROC-AUC  Brier-ish   mean p  mode
   0   23032    16   0.000695    0.0064    0.8229     0.0060  0.00268  full_model_stack
   1   16830    10   0.000594    0.4665    0.8758     0.0039  0.00073  full_model_stack
   2    1574    10   0.006353    0.0202    0.7634     0.0508  0.00020  scorecard_and_rules_only
   3    1412    16   0.011331    0.0270    0.6711     0.0797  0.00087  scorecard_and_rules_only
   4     872     4   0.004587    0.7517    0.8315     0.0153  0.00865  full_model_stack

mean across the 5 scored folds: PR-AUC=0.2543 (min 0.0064, max 0.7517)  ROC-AUC=0.7929 (min 0.6711, max 0.8758)
  -> THIS is the headline shape: an unweighted mean of per-fold readings, with the
     spread shown because a fold holding 4 positives cannot carry a 0.75 the way a
     fold holding 16 can. The plan's §18 leakage trigger (~0.9 PR-AUC on a first
     honest run) is not approached, which is what an honest first run looks like.

A pooled-across-folds fig

## 2. The calibration refusal, in its own words

Plan §10 and `config/model.yaml` refuse calibration below a positive count rather than fitting a
curve on noise. When that fires, the row must *say* it did, with the count and the floor — a
silent degradation to an uncalibrated number is the failure mode this guards against.


In [3]:
from oxbow.config import load_yaml

cal = load_yaml(ROOT / "config" / "model.yaml")["calibration"]
print(f"min_positives_for_calibration : {cal['min_positives_for_calibration']}")
print(f"isotonic_min_positives        : {cal['isotonic_min_positives']}")
print()
print(test_rows.group_by("fold", "calibrated").agg(
    pl.len().alias("rows"),
    pl.col(LABEL).sum().alias("positives_in_test"),
    pl.col("uncalibrated_reason").first().str.slice(0, 78).alias("reason (first 78)"),
).sort("fold"))


min_positives_for_calibration : 50
isotonic_min_positives        : 500

shape: (5, 5)
┌──────┬────────────┬───────┬───────────────────┬─────────────────────────────────┐
│ fold ┆ calibrated ┆ rows  ┆ positives_in_test ┆ reason (first 78)               │
│ ---  ┆ ---        ┆ ---   ┆ ---               ┆ ---                             │
│ i32  ┆ bool       ┆ u32   ┆ i64               ┆ str                             │
╞══════╪════════════╪═══════╪═══════════════════╪═════════════════════════════════╡
│ 0    ┆ false      ┆ 23032 ┆ 16                ┆ probabilities are uncalibrated… │
│ 1    ┆ false      ┆ 16830 ┆ 10                ┆ probabilities are uncalibrated… │
│ 2    ┆ false      ┆ 1574  ┆ 10                ┆ probabilities are uncalibrated… │
│ 3    ┆ false      ┆ 1412  ┆ 16                ┆ probabilities are uncalibrated… │
│ 4    ┆ false      ┆ 872   ┆ 4                 ┆ probabilities are uncalibrated… │
└──────┴────────────┴───────┴───────────────────┴─────────────────────────

In [4]:
reasons = test_rows.get_column("uncalibrated_reason").drop_nulls().unique().to_list()
print(f"distinct uncalibrated reasons: {len(reasons)}")
for reason in reasons[:3]:
    print("  -", reason[:200])
print()
calibrated_any = bool(test_rows.get_column("calibrated").any())
print("any test row reports calibrated=True:", calibrated_any)
print("  -> when False, MODEL_CARD's calibration section must report the refusal, not a curve.")
print("     DEV-024 records the arithmetic: the 40k slice carries 108 positives total against a")
print("     validation floor of 50, so every fold is below it by count, not by defect.")


distinct uncalibrated reasons: 4
  - probabilities are uncalibrated: 6 validation positives is below calibration.min_positives_for_calibration=50; an observed rate measured on that few rows is noise, and Module C multiplies it by money. 
  - probabilities are uncalibrated: 14 validation positives is below calibration.min_positives_for_calibration=50; an observed rate measured on that few rows is noise, and Module C multiplies it by money.
  - probabilities are uncalibrated: 18 validation positives is below calibration.min_positives_for_calibration=50; an observed rate measured on that few rows is noise, and Module C multiplies it by money.

any test row reports calibrated=True: False
  -> when False, MODEL_CARD's calibration section must report the refusal, not a curve.
     DEV-024 records the arithmetic: the 40k slice carries 108 positives total against a
     validation floor of 50, so every fold is below it by count, not by defect.


## 3. Every score carries its explanation, or names why it does not

SHAP is computed during the run and stored on the row. A row whose explanation was skipped must
carry a reason — an absent explanation with no explanation for the absence is the shape of a bug
that reads as a feature.


In [5]:
coverage = scored.group_by("scoring_mode").agg(
    pl.len().alias("rows"),
    pl.col("shap_json").is_not_null().sum().alias("shap_present"),
    pl.col("shap_json").str.len_bytes().mean().round(0).alias("mean shap bytes"),
    pl.col("explanation_source").mode().first().alias("modal source"),
)
print(coverage)
print()
missing = scored.filter(pl.col("shap_json").is_null())
print(f"rows without shap_json: {missing.height}")
if missing.height:
    print(missing.group_by("explanation_source", "explanation_fallback_reason").agg(
        pl.len().alias("rows")
    ).sort("rows", descending=True).head(6))
print()
base = scored.get_column("shap_base_value")
print(f"shap_base_value: nulls={base.null_count()} min={base.min():.6f} max={base.max():.6f}")
print("  -> an explanation must sum back to the score it explains; the base value is the")
print("     additive identity the packet renders, so a null base is an unverifiable story.")


shape: (2, 5)
┌──────────────────────────┬───────┬──────────────┬─────────────────┬─────────────────────┐
│ scoring_mode             ┆ rows  ┆ shap_present ┆ mean shap bytes ┆ modal source        │
│ ---                      ┆ ---   ┆ ---          ┆ ---             ┆ ---                 │
│ str                      ┆ u32   ┆ u32          ┆ f64             ┆ str                 │
╞══════════════════════════╪═══════╪══════════════╪═════════════════╪═════════════════════╡
│ full_model_stack         ┆ 40734 ┆ 40734        ┆ 4145.0          ┆ shap-tree-explainer │
│ scorecard_and_rules_only ┆ 2986  ┆ 2986         ┆ 1983.0          ┆ scorecard-explained │
└──────────────────────────┴───────┴──────────────┴─────────────────┴─────────────────────┘

rows without shap_json: 0

shap_base_value: nulls=0 min=-38.547890 max=700.000000
  -> an explanation must sum back to the score it explains; the base value is the
     additive identity the packet renders, so a null base is an unverifiable story.


## 4. The harness's own table

The published ablation table, its leakage control and the tail-risk figures come from
`oxbow backtest`. Read from the artifact when it exists; named as absent when it does not.


In [6]:
import glob

candidates = sorted(glob.glob(str(ROOT / "out" / "backtest" / "**" / "ablation_results.json"),
                             recursive=True))
print("ablation artifacts on disk:", len(candidates))
fresh = None
for path in candidates:
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    corpus = payload.get("corpus", "?")
    fake = "fake" in str(corpus)
    print(f"   {Path(path).parent.name[:34]:<36} corpus={corpus:<26} "
          f"provenance={'FAKE HARNESS' if fake else 'real'}")
    if not fake and fresh is None:
        fresh = (path, payload)

if fresh is None:
    print()
    print("NOT YET LANDED: no real-corpus ablation table exists. Until one does,")
    print("MODEL_CARD.md and ECONOMICS_CARD.md quote the fake harness and must say so.")
    print("Produce it with:")
    print("  uv run oxbow backtest --corpus out/score/%s/backtest_corpus.parquet" % RUN_ID)
else:
    path, payload = fresh
    variants = payload.get("variants") or []
    print(f"\nusing {path}")
    print(f"{'variant':<44} {'PR-AUC':>9} {'notes'}")
    for row in variants[:10]:
        name = str(row.get("label") or row.get("variant") or "?")[:42]
        pr = row.get("pr_auc")
        note = str(row.get("note", ""))[:36]
        print(f"{name:<44} {(f'{pr:.4f}' if isinstance(pr, (int, float)) else '--'):>9} {note}")
    control = payload.get("leakage_control") or {}
    if control:
        print(f"\nleakage control: detected={control.get('detected')} "
              f"control_pr_auc={control.get('control_pr_auc')} "
              f"best_honest_pr_auc={control.get('best_honest_pr_auc')}")
        print("  -> the harness is only trustworthy if that control can fail; a table with no "
              "leaking row proves nothing about leakage.")


ablation artifacts on disk: 3
   01M3EH7VFH4PZZKW0AT0QPZK8X           corpus=paysim-fake+ibm-fake       provenance=FAKE HARNESS
   01M3FSH94NDD93SHDZ720V2RHC           corpus=paysim-fake+ibm-fake       provenance=FAKE HARNESS
   backtest                             corpus=paysim-fake+ibm-fake       provenance=FAKE HARNESS

NOT YET LANDED: no real-corpus ablation table exists. Until one does,
MODEL_CARD.md and ECONOMICS_CARD.md quote the fake harness and must say so.
Produce it with:
  uv run oxbow backtest --corpus out/score/01M3H8WG436R394NZT2GS1KG69/backtest_corpus.parquet


## 5. What is not established here

* **AUC on ~0.13 % positives is a ranking statistic with a wide interval.** No confidence interval
  is computed in this notebook; the harness's tail-risk figures and `LIMITATIONS.md` carry that.
* **The folds are not independent.** Expanding windows share every earlier event, and the last
  fold's 872 test rows are a thin read on a period shift.
* **PaySim's `isFraud` is a simulator's opinion.** Discriminating it well is not evidence about
  real laundering; `MODEL_CARD.md` states this wherever the figure appears.
* **Nothing here measures stability under drift** beyond the PSI the score stage reports.
